# Notebook 2: Uncertainty, two ways (solutions)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/centre-for-humanities-computing/introduction-bayesian-statistics/blob/master/notebooks/solutions/02_uncertainty_two_ways_solutions.ipynb)

**Session 1 - Linear regressions** · Block 2

In this notebook you will:

1. see where a **standard error** comes from, by simulating many studies,
2. check what the 95% of a **confidence interval** means,
3. compute a **p-value** by simulation,
4. fit the same model the **Bayesian** way with PyMC,
5. read a **posterior**: plausible lines, credible intervals, probabilities,
6. see **when** the two approaches disagree.

> **How to use this notebook.** Run the cells from top to bottom with **Shift + Enter**.
> Cells marked ✏️ ask you to complete or change something. If something breaks,
> use *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).

In [ ]:
# @title ⚙️ Setup: run this cell first (Shift + Enter)
import importlib.util
import subprocess
import sys
import warnings

PACKAGES = ["pymc", "arviz", "statsmodels"]
missing = [p for p in PACKAGES if importlib.util.find_spec(p) is None]
if missing:
    print("Installing", *missing, "... (about a minute)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import arviz as az
import pymc as pm

# Plot colours
BLUE, ROSE, GREEN, GREY = "#5B7FA6", "#BC6D86", "#5E8C4F", "#595959"
plt.rcParams.update(
    {
        "figure.figsize": (7, 4.5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#EDEDED",
        "font.size": 12,
    }
)
print("Ready!")

## 1. Back to our data

Same simulated reading-time data as in Notebook 1 (true intercept 0.5, slope 0.8, σ = 1).

In [ ]:
def simulate_reading(n=40, intercept=0.5, slope=0.8, sigma=1.0, seed=226):
    """Simulate n texts: length (hundreds of words) and reading time (minutes)."""
    rng = np.random.default_rng(seed)
    words = rng.uniform(1, 12, size=n).round(1)
    time = intercept + slope * words + rng.normal(0, sigma, size=n)
    return pd.DataFrame({"words": words, "time": time.round(2)})


df = simulate_reading()
df.head()

In [ ]:
fit = smf.ols("time ~ words", data=df).fit()
fit.summary().tables[1]

# Part A: the frequentist answer

## 2. Where does the standard error come from?

Imagine running the same study again and again: the **same 40 texts**, read by new
participants. The reading times change, so each study gives a slightly different slope.
Because we know the true line, we can **simulate** this. `repeat_study()` keeps our 40 texts
and draws new reading times:

🎬 **Repeating the study.** Each flash is a new run of the study (same 40 texts, new reading times); its least-squares line (blue) leaves a trace and its slope drops onto the histogram. The width of that histogram (bracket: ±1 SD) is the standard error. The code below does the same simulation.

![Animation: 200 repeated studies, their least-squares lines and a histogram of their slopes](https://raw.githubusercontent.com/centre-for-humanities-computing/introduction-bayesian-statistics/master/animations/renders/02_repeat_the_study.gif)

In [ ]:
def repeat_study(slope=0.8, seed=None):
    """Repeat our study: the same 40 texts, new reading times."""
    rng = np.random.default_rng(seed)
    time = 0.5 + slope * df["words"] + rng.normal(0, 1.0, size=len(df))
    return pd.DataFrame({"words": df["words"], "time": time.round(2)})

In [ ]:
rng = np.random.default_rng(7)
slopes = []
for _ in range(2000):
    new_df = repeat_study(seed=int(rng.integers(1e9)))
    new_fit = smf.ols("time ~ words", data=new_df).fit()
    slopes.append(new_fit.params["words"])
slopes = np.array(slopes)

fig, ax = plt.subplots()
ax.hist(slopes, bins=40, color=BLUE, alpha=0.8)
ax.axvline(0.8, color=GREEN, lw=2.5, ls="--", label="true slope")
ax.set(
    xlabel="Estimated slope", ylabel="Number of studies", title="2000 simulated studies"
)
ax.legend()
plt.show()

✏️ **Exercise 2.1.** The **standard error** is the standard deviation of this histogram.
Compute it with `np.std(slopes)` and compare it with the `std err` that `statsmodels` estimated
from our **single** dataset (`fit.bse["words"]`).

In [ ]:
print("SD of the 2000 simulated slopes:", round(np.std(slopes), 3))
print("standard error from statsmodels:", round(fit.bse["words"], 3))

✅ **Answer.** They are almost the same (about 0.054 and 0.053; the small difference is
simulation noise). From **one** dataset, `statsmodels` estimates how much the slope would vary
from study to study, without repeating the study.

## 3. What does "95% confidence" mean?

The 95% confidence interval of our slope:

In [ ]:
fit.conf_int().loc["words"]

The 95% is a property of the **procedure**: if we repeated the study many times and built an
interval each time, about 95% of those intervals would contain the true slope. Let's check:

In [ ]:
rng = np.random.default_rng(11)
contains_truth = []
for _ in range(200):
    new_df = repeat_study(seed=int(rng.integers(1e9)))
    low, high = smf.ols("time ~ words", data=new_df).fit().conf_int().loc["words"]
    contains_truth.append(low <= 0.8 <= high)
contains_truth = np.array(contains_truth)
contains_truth[:10]

✏️ **Exercise 3.1.** What share of the 200 intervals contain the true slope?
(Hint: the mean of a list of `True`/`False` values is the share of `True`.)

In [ ]:
contains_truth.mean()

✅ **Answer.** About 0.95 (exactly 0.95 here; with only 200 studies the share wobbles from run to run).
Our *own* interval either contains 0.8 or it does not; we cannot
say "there is a 95% probability that the true slope is in our interval". That reading belongs
to the Bayesian credible interval (Part B).

## 4. p-values by simulation

The p-value answers: *if the true slope were 0 (the "null world"), how often would we see a
slope at least as far from 0 as ours?*

In [ ]:
rng = np.random.default_rng(5)
null_slopes = []
for _ in range(1000):
    null_df = repeat_study(slope=0.0, seed=int(rng.integers(1e9)))
    null_slopes.append(smf.ols("time ~ words", data=null_df).fit().params["words"])
null_slopes = np.array(null_slopes)

our_slope = fit.params["words"]
fig, ax = plt.subplots()
ax.hist(null_slopes, bins=40, color="#BFBFBF", label="slopes in the null world")
ax.axvline(our_slope, color=BLUE, lw=3, label="our slope")
ax.set(xlabel="Estimated slope", ylabel="Number of studies")
ax.legend()
plt.show()

print(
    "share of null slopes at least as extreme as ours:",
    np.mean(np.abs(null_slopes) >= abs(our_slope)),
)
print("p-value reported by statsmodels:", fit.pvalues["words"])

Our slope is far outside what the null world produces, so p is tiny (statsmodels prints
`0.000`). Remember: a p-value is **not** the probability that the null hypothesis is true, and
a small p-value does not mean the effect is large or important.

✏️ **Exercise 4.1.** Simulate **one** study where the true slope is small but not zero
(`slope=0.05`) and fit it. Is the p-value below 0.05? What does this tell you about
"not significant"?

In [ ]:
small_df = simulate_reading(slope=0.05, seed=3)
small_fit = smf.ols("time ~ words", data=small_df).fit()
print("estimated slope:", round(small_fit.params["words"], 3))
print("p-value:", round(small_fit.pvalues["words"], 3))

✅ **Answer.** Here p ≈ 0.26: "not significant", although the true slope is *not* zero. A large
p-value means the data are compatible with a slope of 0, **not** that the slope is 0. Small
effects need more data to be detected.

# Part B: the Bayesian answer

## 5. Writing the model in PyMC

The Bayesian model has the same likelihood as least squares, plus **priors**: what we consider
plausible *before* seeing the data.

| | maths | meaning |
|---|---|---|
| likelihood | time ~ Normal(a + b · words, σ) | points are scattered around the line |
| prior | a ~ Normal(0, 5) | intercept: a few minutes, give or take |
| prior | b ~ Normal(0, 2) | slope: a few minutes per 100 words at most |
| prior | σ ~ HalfNormal(2) | scatter: positive, a few minutes at most |

`pm.sample()` then draws samples from the **posterior** with MCMC. The first run takes a while
(30 to 60 seconds) because PyMC compiles the model.

In [ ]:
with pm.Model() as model:
    # priors
    intercept = pm.Normal("intercept", mu=0, sigma=5)
    slope = pm.Normal("slope", mu=0, sigma=2)
    sigma = pm.HalfNormal("sigma", sigma=2)

    # likelihood
    mu = intercept + slope * df["words"]
    pm.Normal("time", mu=mu, sigma=sigma, observed=df["time"])

    idata = pm.sample(draws=1000, tune=1000, chains=2, random_seed=1)

First, a quick health check: the column **`r_hat`** should be **1.00** (or at most 1.01) for
every parameter, meaning the independent chains agree.

In [ ]:
az.summary(idata, var_names=["intercept", "slope", "sigma"])

## 6. The posterior: many plausible values

The posterior is a big bag of **draws**: 2 chains × 1000 draws = 2000 plausible values for each
parameter. Let's take them out as plain arrays:

In [ ]:
intercept_draws = idata.posterior["intercept"].values.ravel()
slope_draws = idata.posterior["slope"].values.ravel()
print(len(slope_draws), "draws, the first five slopes:", slope_draws[:5].round(3))

fig, ax = plt.subplots()
ax.hist(slope_draws, bins=40, color=ROSE, alpha=0.85)
ax.set(
    xlabel="Slope (minutes per 100 words)",
    ylabel="Number of draws",
    title="Posterior of the slope",
)
plt.show()

Each draw (intercept, slope) is a whole line. Let's plot 100 of them:

In [ ]:
xs = np.array([0, 12.5])
fig, ax = plt.subplots()
for i in range(0, 2000, 20):
    ax.plot(xs, intercept_draws[i] + slope_draws[i] * xs, color=ROSE, alpha=0.1)
ax.plot(
    xs,
    fit.params["Intercept"] + fit.params["words"] * xs,
    color=BLUE,
    ls="--",
    label="least-squares line",
)
ax.scatter(df["words"], df["time"], color=GREY, zorder=3)
ax.set(
    xlabel="Text length (hundreds of words)",
    ylabel="Reading time (minutes)",
    xlim=(0, 12.5),
    ylim=(0, 12),
    title="100 plausible lines from the posterior",
)
ax.legend()
plt.show()

✏️ **Exercise 6.1.** Using `slope_draws`:

1. compute the **95% credible interval**: the middle 95% of the draws, i.e. the 2.5% and 97.5%
   quantiles (`np.quantile(slope_draws, [0.025, 0.975])`);
2. compute **P(slope > 0.8 | data)**: the share of draws above 0.8.

In [ ]:
credible = np.quantile(slope_draws, [0.025, 0.975])
p_above = (slope_draws > 0.8).mean()
print("95% credible interval:", credible.round(2))
print("P(slope > 0.8 | data) =", round(p_above, 2))

✅ **Answer.** The credible interval is about [0.68, 0.89] and P(slope > 0.8) ≈ 0.41. Given the
model and the priors, there is a 95% probability that the slope lies in that interval: a
direct statement about the slope, which a confidence interval cannot make.

## 7. Side by side

In [ ]:
ci_low, ci_high = fit.conf_int().loc["words"]
cr_low, cr_high = np.quantile(slope_draws, [0.025, 0.975])
pd.DataFrame(
    {
        "frequentist (statsmodels)": [fit.params["words"], ci_low, ci_high],
        "Bayesian (PyMC)": [slope_draws.mean(), cr_low, cr_high],
    },
    index=["estimate", "95% interval: low", "95% interval: high"],
).round(3)

With weak priors and 40 data points, the numbers are almost identical. What differs is the
**interpretation**.

## 8. When do the two approaches disagree?

A colleague is convinced that people read 250 words per minute, a typical speed for casual
silent reading, so a slope of **0.4 minutes per 100 words**, and is quite sure: prior
`slope ~ Normal(0.4, 0.1)`. But our participants read carefully, for a comprehension test,
which is much slower (125 words per minute in our simulation): the colleague's belief comes
from a different situation.
The helper below fits the Bayesian model with any prior on the slope:

In [ ]:
def fit_bayes(data, slope_mu=0.0, slope_sigma=2.0):
    """Fit the reading-time model with PyMC and return the slope draws."""
    with pm.Model():
        intercept = pm.Normal("intercept", mu=0, sigma=5)
        slope = pm.Normal("slope", mu=slope_mu, sigma=slope_sigma)
        sigma = pm.HalfNormal("sigma", sigma=2)
        mu = intercept + slope * data["words"]
        pm.Normal("time", mu=mu, sigma=sigma, observed=data["time"])
        idata = pm.sample(
            draws=1000, tune=1000, chains=2, random_seed=2, progressbar=False
        )
    return idata.posterior["slope"].values.ravel()

✏️ **Exercise 8.1.** With only **8 texts** (`simulate_reading(n=8, seed=21)`), compare:

* the least-squares slope,
* the posterior mean of the slope with the strong prior `slope_mu=0.4, slope_sigma=0.1`.

Then try with **400 texts** (`n=400, seed=33`). What happens?

In [ ]:
for n, seed in [(8, 21), (400, 33)]:
    data = simulate_reading(n=n, seed=seed)
    ols_slope = smf.ols("time ~ words", data=data).fit().params["words"]
    draws = fit_bayes(data, slope_mu=0.4, slope_sigma=0.1)
    print(
        f"n = {n:3d}: least squares = {ols_slope:.2f}, "
        f"posterior mean (strong prior) = {draws.mean():.2f}"
    )

✅ **Answer.** With 8 texts, the strong prior pulls the estimate towards 0.4 (≈ 0.48 instead of
≈ 0.59). With 400 texts, the data overwhelm the prior and both approaches agree (≈ 0.8).
Priors matter when data are scarce: state them, justify them (does the prior come from a
comparable situation?), and check how much they matter.

## Summary

| | frequentist | Bayesian |
|---|---|---|
| uncertainty from… | imagined repeated studies | the posterior distribution |
| 95% interval means… | 95% of such intervals contain the truth | 95% probability the slope is inside |
| extra ingredient | — | priors |
| tools | `smf.ols(...)`, `conf_int()`, `pvalues` | `pm.Model()`, `pm.sample()`, `az.summary()` |

➡️ Next: **more than one predictor** (Notebook 3)